# Stress test 2 — iterative / streaming pipeline (deep chains)

Incremental workflows (streaming batches, RL checkpoints, online learning) produce long linear lineages. This stresses node-creation throughput and the recursive `prune`.

In [1]:
import shutil
import os
import time
import ancestree
from ancestree import LineageStore

SCRATCH = "_stress"  # all stores written here; safe to delete
shutil.rmtree(SCRATCH, ignore_errors=True)
os.makedirs(SCRATCH, exist_ok=True)


def store(name, **kw):
    return LineageStore(f"{SCRATCH}/{name}", **kw)


def node_dirs(s):
    return [d for d in os.listdir(s.root) if not d.startswith(".")]


print("ancestree", getattr(ancestree, "__version__", "?"))

ancestree 0.1.0


## ⚠️ Finding #2 — node creation is dominated by git provenance (~16 ms/node)

Every `create_node` shells out to `git` three times to record provenance. On bulk pipelines this dominates — 40× the cost of the actual store work.

In [2]:
s = store("stream", chunk=True)
prev = None
t = time.time()
with s.create_node(step_type="batch") as n0:
    n0.add_meta("batch", 0)
prev = n0.node_id
B = 300
for i in range(1, B):
    with s.create_node(step_type="batch", parent=prev) as n:
        (n / "state.bin").write_bytes(os.urandom(2000))
        n.add_meta("batch", i)
    prev = n.node_id
dt = time.time() - t
print(f"{B} sequential nodes in {dt:.1f}s  ->  {dt / B * 1000:.1f} ms/node")
print(
    "For a 100k-step pipeline that is ~",
    round(dt / B * 100_000 / 60),
    "minutes just in creation.",
)

300 sequential nodes in 6.1s  ->  20.3 ms/node
For a 100k-step pipeline that is ~ 34 minutes just in creation.


In [3]:
# get_lineage is iterative — fine even on deep chains
lin = s.get_lineage(prev)
print("lineage depth:", len(lin))

lineage depth: 300


## ✅ Finding #3 (FIXED) — `prune` on chains deeper than ~1000

`prune` used to recurse one frame per descendant and raise `RecursionError` past the interpreter limit (default 1000) — for both the real delete *and* the dry-run preview. It now walks the subtree **iteratively**, so arbitrarily deep lineages prune and preview cleanly (deepest-first, target last).

In [4]:
# Build a chain well past the recursion limit
deep = store("deep", chunk=False)
with deep.create_node(step_type="root") as r:
    r.add_meta("k", 0)
prev = r.node_id
for i in range(1100):
    with deep.create_node(step_type="s", parent=prev) as n:
        n.add_meta("i", i)
    prev = n.node_id

preview = deep.prune(r.node_id, dry_run=True)  # no RecursionError now
print(f"dry-run preview of a {len(preview)}-deep chain: ok")
print(
    "deepest-first:",
    preview[0].node_id == prev,
    "| target last:",
    preview[-1].node_id == r.node_id,
)
deleted = deep.prune(r.node_id, dry_run=False)
print(f"pruned {len(deleted)} nodes; remaining: {len(node_dirs(deep))}")

dry-run preview of a 1101-deep chain: ok
deepest-first: True | target last: True
pruned 1101 nodes; remaining: 0
